# Notebook 4 — Solutions: Advanced Connectome Analysis

**BINFX410 · Chapter 10 · Connectomics**

Reference solutions for the five exercises in `04_connectome_analysis.ipynb`.  
Run notebooks 1–4 first to generate `../data/connectome_architecture.pkl`.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pandas as pd

DATA_DIR = Path('../data')

with open(DATA_DIR / 'connectome_architecture.pkl', 'rb') as f:
    arch = pickle.load(f)

with open(DATA_DIR / 'connectome_graphs.pkl', 'rb') as f:
    graph_data = pickle.load(f)

A       = arch['adjacency_matrix']    # (N, N)
P       = arch['transition_matrix']   # row-normalised A
G       = arch['G']                   # detected DiGraph
G_gt    = graph_data['G_gt']          # ground-truth DiGraph
N       = arch['n_neurons']
nodes   = arch['nodes']
image   = graph_data['image']

A_gt    = nx.to_numpy_array(G_gt, nodelist=sorted(G_gt.nodes()), weight='weight')

print(f'Detected connectome: {N} neurons, {G.number_of_edges()} edges')
print(f'Adjacency matrix shape: {A.shape}')

---
## Exercise 4.1 — Adjacency Matrix Symmetry

**Task:** Compute `(A + A^T) / 2`. What does symmetry mean biologically?

In [ ]:
A_sym = (A + A.T) / 2   # symmetrised adjacency matrix

# Quantify asymmetry
asym_score = np.linalg.norm(A - A.T, 'fro') / (np.linalg.norm(A, 'fro') + 1e-8)
print(f'Asymmetry score (Frobenius): {asym_score:.4f}')
print(f'  0 = perfectly symmetric, 1 = maximally asymmetric')
print()

# Count reciprocal vs. unidirectional edges
reciprocal   = sum(1 for u, v in G.edges() if G.has_edge(v, u))
unidirectional = G.number_of_edges() - reciprocal
print(f'Reciprocal edges (A→B AND B→A) : {reciprocal // 2}')
print(f'Unidirectional edges            : {unidirectional}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
node_labels = [f'N{n}' for n in nodes]

for ax, mat, title in [
    (axes[0], A,     'Original A (asymmetric)'),
    (axes[1], A.T,   'Transposed A^T'),
    (axes[2], A_sym, '(A + A^T)/2 (symmetric)'),
]:
    vmax = max(A.max(), 0.01)
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Synapse weight')
    ax.set_xticks(range(N)); ax.set_yticks(range(N))
    ax.set_xticklabels(node_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(node_labels, fontsize=8)
    ax.set_title(title, fontsize=12)

plt.suptitle('Exercise 4.1 — Adjacency Matrix Symmetry', fontsize=13)
plt.tight_layout()
plt.show()

print()
print('Biological interpretation of symmetry:')
print('  A[i,j] = weight of synapse from i to j (directed).')
print('  Symmetry means every connection i→j is exactly mirrored by j→i.')
print('  Biological significance: fully symmetric connectivity implies')
print('    MUTUAL EXCITATION: every pair of connected neurons excites each other.')
print('    This topology is associated with winner-take-all circuits and attractors.')
print('  Real connectomes are nearly always ASYMMETRIC — directionality encodes')
print('  information flow: sensory → interneuron → motor paths are one-directional.')
print()
print('  (A + A^T)/2 is the undirected weighted representation: it treats each edge')
print('  as bidirectional with the mean strength, useful for community detection')
print('  algorithms that require undirected graphs.')

---
## Exercise 4.2 — Eigenvalue Spectrum of the Adjacency Matrix

**Task:** Compute the eigenvalue spectrum. How does the spectral radius relate to signal amplification? Compare detected vs. GT.

In [ ]:
# Compute eigenvalues for detected and GT adjacency matrices
eigenvalues_det = np.linalg.eigvals(A)
eigenvalues_gt  = np.linalg.eigvals(A_gt)

spectral_radius_det = np.abs(eigenvalues_det).max()
spectral_radius_gt  = np.abs(eigenvalues_gt).max()

print('Eigenvalue Spectrum Analysis')
print('=' * 40)
print(f'\nDetected connectome:')
print(f'  Spectral radius ρ(A) = {spectral_radius_det:.4f}')
print(f'  Largest real eigenvalue: {eigenvalues_det.real.max():.4f}')
print(f'  All eigenvalues: {np.sort(np.abs(eigenvalues_det))[::-1].round(4)}')

print(f'\nGround-truth connectome:')
print(f'  Spectral radius ρ(A) = {spectral_radius_gt:.4f}')
print(f'  Largest real eigenvalue: {eigenvalues_gt.real.max():.4f}')

# Visualize spectrum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, eigs, title, color in [
    (axes[0], eigenvalues_det, 'Detected Connectome', 'steelblue'),
    (axes[1], eigenvalues_gt,  'Ground-Truth Connectome', 'tomato'),
]:
    ax.scatter(eigs.real, eigs.imag, s=80, color=color, edgecolors='black', zorder=5)
    circle = plt.Circle((0, 0), np.abs(eigs).max(), fill=False, linestyle='--', color='gray')
    ax.add_patch(circle)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_aspect('equal')
    ax.set_xlabel('Re(λ)'); ax.set_ylabel('Im(λ)')
    ax.set_title(f'{title}\nSpectral radius = {np.abs(eigs).max():.3f}', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Exercise 4.2 — Adjacency Matrix Eigenvalue Spectrum', fontsize=13)
plt.tight_layout()
plt.show()

# Demonstrate signal amplification via repeated multiplication
print()
print('Signal amplification demonstration:')
x = np.ones(N) / N  # uniform initial signal
for t in [1, 2, 5, 10]:
    x_t = np.linalg.matrix_power(A, t) @ (np.ones(N) / N)
    norm = np.linalg.norm(x_t)
    print(f'  ||A^{t:2d} * x||_2 = {norm:.4f}')
print()
print('Interpretation:')
print('  The spectral radius ρ(A) determines long-term signal dynamics:')
print('  ρ < 1 → signals decay to zero (stable, no runaway activity)')
print('  ρ = 1 → signals persist (marginally stable)')
print('  ρ > 1 → signals amplify exponentially (unstable without inhibition)')
print(f'  Our detected connectome has ρ={spectral_radius_det:.3f}, so signals')
if spectral_radius_det < 1:
    print('  will eventually decay without sustained input.')
elif spectral_radius_det == 1:
    print('  will persist at constant amplitude.')
else:
    print('  will amplify — the network needs inhibition for stability.')

---
## Exercise 4.3 — Leaky Integrate-and-Fire (LIF) Simulation

**Task:** Simulate `act_new[i] = 0.8 * act[i] + sum_j P[j,i] * act[j]` for 20 steps from the highest out-degree neuron. Does it reach steady state?

In [ ]:
DECAY = 0.8
N_STEPS = 20

# Find seed neuron (highest out-degree)
out_degrees  = dict(G.out_degree())
seed_neuron  = max(out_degrees, key=out_degrees.get) if out_degrees else 0
print(f'Seed neuron: N{seed_neuron} (out-degree={out_degrees.get(seed_neuron, 0)})')

# Initial condition: unit pulse at seed
act = np.zeros(N)
act[seed_neuron] = 1.0

activation_history = [act.copy()]
total_activity     = [act.sum()]

# LIF dynamics: act_new[i] = decay * act[i] + sum_j P[j,i] * act[j]
# In matrix form: act_new = decay * act + P^T @ act = (decay * I + P^T) @ act
M = DECAY * np.eye(N) + P.T   # effective recurrence matrix

for t in range(N_STEPS):
    act = M @ act
    # Optional: clamp to [0, inf] for biological plausibility (no negative firing rates)
    act = np.clip(act, 0, None)
    activation_history.append(act.copy())
    total_activity.append(act.sum())

act_matrix = np.array(activation_history)  # (T+1, N)

# Check for steady state: compare last few steps
delta_last5 = [np.linalg.norm(activation_history[t+1] - activation_history[t])
               for t in range(N_STEPS-5, N_STEPS)]
print(f'||act[t+1] - act[t]|| for last 5 steps: {[f"{d:.5f}" for d in delta_last5]}')
print(f'Converged (delta < 1e-4): {all(d < 1e-4 for d in delta_last5)}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
im = axes[0].imshow(act_matrix.T, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=axes[0], label='Activation')
axes[0].set_yticks(range(N))
axes[0].set_yticklabels([f'N{n}' for n in nodes])
axes[0].set_xlabel('Time step')
axes[0].set_title(f'LIF Dynamics (decay={DECAY})\nSeed: N{seed_neuron}', fontsize=12)

# Total activity curve
axes[1].plot(total_activity, 'o-', color='steelblue')
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Total network activation (sum)')
axes[1].set_title('Total Activation Over Time', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Exercise 4.3 — Leaky Integrate-and-Fire Simulation', fontsize=13)
plt.tight_layout()
plt.show()

# Theoretical analysis
M_spectral_radius = np.abs(np.linalg.eigvals(M)).max()
print(f'\nSpectral radius of M = decay*I + P^T: {M_spectral_radius:.4f}')
print()
print('Steady-state analysis:')
print(f'  ρ(M) = {M_spectral_radius:.4f}')
if M_spectral_radius < 1:
    print('  ρ < 1 → network converges to zero (all activity decays).')
    print('  With decay=0.8 and a sparse P, the decay term dominates.')
elif M_spectral_radius == 1:
    print('  ρ = 1 → network maintains constant total activity.')
else:
    print('  ρ > 1 → activity diverges without additional normalization.')
    print('  In a real neuron model, nonlinearities (firing threshold, saturation)')
    print('  would prevent unbounded growth.')

---
## Exercise 4.4 — Louvain Community Detection

**Task:** Detect functional modules. Visualize with nodes colored by community. Are communities spatially clustered?

In [ ]:
from networkx.algorithms import community as nx_comm

G_undet = G.to_undirected()

if G_undet.number_of_edges() == 0:
    print('No edges — cannot detect communities. Using single community.')
    louvain_comm = [set(G_undet.nodes())]
else:
    # Louvain (available in networkx >= 3.0 via nx.community.louvain_communities)
    try:
        louvain_comm = list(nx_comm.louvain_communities(G_undet, seed=42))
        method = 'Louvain'
    except AttributeError:
        # Fall back to greedy modularity if Louvain unavailable
        louvain_comm = list(nx_comm.greedy_modularity_communities(G_undet))
        method = 'Greedy Modularity (Louvain unavailable in this networkx version)'

    print(f'Community detection method: {method}')
    print(f'Number of communities: {len(louvain_comm)}')

node_to_comm = {}
for ci, comm in enumerate(louvain_comm):
    for node in comm:
        node_to_comm[node] = ci

for ci, comm in enumerate(louvain_comm):
    print(f'  Community {ci}: neurons {sorted(comm)}')

# Compute modularity
if G_undet.number_of_edges() > 0:
    Q = nx_comm.modularity(G_undet, louvain_comm)
    print(f'\nModularity Q = {Q:.4f}  (higher is better; random graph ≈ 0)')

# ── Visualization ─────────────────────────────────────────────────────────────
cmap = plt.cm.get_cmap('Set2', len(louvain_comm))
node_colors_graph = [cmap(node_to_comm.get(n, 0)) for n in G.nodes()]

detected_soma = graph_data['detected_soma']

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Graph view
pos = nx.get_node_attributes(G, 'pos')
if not pos: pos = nx.spring_layout(G, seed=42)
nx.draw_networkx_nodes(G, pos, ax=axes[0], node_color=node_colors_graph,
                       node_size=500, edgecolors='black', linewidths=1)
nx.draw_networkx_edges(G, pos, ax=axes[0], edge_color='gray', alpha=0.6,
                       arrows=True, arrowsize=12)
nx.draw_networkx_labels(G, pos, {n: f'N{n}' for n in G.nodes()}, ax=axes[0], font_size=9)

# Community legend
import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=cmap(i), label=f'Community {i}') for i in range(len(louvain_comm))]
axes[0].legend(handles=patches, loc='upper right')
axes[0].set_title('Communities in Connectome Graph', fontsize=12)
axes[0].axis('off')

# Spatial view
axes[1].imshow(image, cmap='gray', vmin=0, vmax=0.8)
for i, (r, c) in enumerate(detected_soma):
    color = cmap(node_to_comm.get(i, 0))
    circle = plt.Circle((c, r), 20, facecolor=color, edgecolor='white',
                         alpha=0.8, linewidth=2)
    axes[1].add_patch(circle)
    axes[1].text(c, r, f'N{i}\nC{node_to_comm.get(i,0)}', ha='center', va='center',
                 fontsize=7, color='black', fontweight='bold')
axes[1].set_title('Communities overlaid on image', fontsize=12)
axes[1].axis('off')

plt.suptitle('Exercise 4.4 — Louvain Community Detection', fontsize=13)
plt.tight_layout()
plt.show()

# Spatial correlation analysis
print('\nSpatial clustering analysis:')
for ci, comm in enumerate(louvain_comm):
    if len(comm) < 2: continue
    comm_nodes = sorted(comm)
    coords = np.array([detected_soma[n] for n in comm_nodes if n < len(detected_soma)])
    if len(coords) >= 2:
        from scipy.spatial.distance import pdist
        intra_dist = pdist(coords).mean()
        all_coords = np.array(detected_soma)
        inter_dist = np.linalg.norm(all_coords - all_coords.mean(axis=0), axis=1).mean()
        print(f'  Community {ci}: mean intra-distance={intra_dist:.1f}px, '
              f'global spread={inter_dist:.1f}px')

---
## Exercise 4.5 (Challenge) — Comparison with the C. elegans Connectome

**Task:** Compare our synthetic connectome against the 302-neuron C. elegans connectome. How do degree distributions compare?

The full C. elegans connectome (302 neurons, ~7,000 chemical synapses) is available from the OpenWorm project at https://github.com/openworm/c302 or as a pre-parsed CSV from multiple sources. Below we load a simplified version from a public URL, or fall back to a synthetic approximation if offline.

In [ ]:
# ── Option A: Load C. elegans data from a CSV file if available ───────────────
# The Varshney et al. (2011) dataset is widely used; download from:
# https://www.wormatlas.org/images/ConnectomicsData.xls
# or the connectome-tools package: pip install connectome-tools

# ── Option B: Approximate C. elegans statistics from the literature ───────────
# We reproduce known properties for comparison:
# Varshney et al. (2011) Chemical synapse connectome:
#   302 neurons, 6393 synaptic connections, density ≈ 0.070
#   Average degree ≈ 42 (in+out)
#   Clustering coefficient ≈ 0.028
#   Average path length ≈ 2.65
#   Power-law degree distribution (scale-free-like tail)

celegans_stats = {
    'n_neurons'      : 302,
    'n_synapses'     : 6393,
    'density'        : 6393 / (302 * 301),
    'avg_total_degree': 6393 * 2 / 302,
    'clustering_coef': 0.028,
    'avg_path_length': 2.65,
    'spectral_radius': 'unknown (requires full matrix)',
}

# Our synthetic connectome statistics
G_undet = G.to_undirected()
lcc     = max(nx.connected_components(G_undet), key=len)
G_lcc   = G_undet.subgraph(lcc)

our_stats = {
    'n_neurons'       : G.number_of_nodes(),
    'n_synapses'      : G.number_of_edges(),
    'density'         : nx.density(G),
    'avg_total_degree': (G.number_of_edges() * 2) / (G.number_of_nodes() + 1e-8),
    'clustering_coef' : nx.average_clustering(G_lcc) if G_lcc.number_of_edges() > 0 else 0,
    'avg_path_length' : nx.average_shortest_path_length(G_lcc) if G_lcc.number_of_edges() > 0 else float('nan'),
    'spectral_radius' : float(np.abs(np.linalg.eigvals(A)).max()),
}

print('Comparison: Our Synthetic Connectome vs. C. elegans (Varshney et al., 2011)')
print('=' * 70)
print(f'{"Property":<25} {"Our connectome":>20} {"C. elegans":>20}')
print('-' * 70)
for key in celegans_stats:
    ours = our_stats.get(key, 'N/A')
    cel  = celegans_stats[key]
    if isinstance(ours, float): ours = f'{ours:.4f}'
    if isinstance(cel, float):  cel  = f'{cel:.4f}'
    print(f'{key:<25} {str(ours):>20} {str(cel):>20}')

# Plot degree distribution comparison
fig, ax = plt.subplots(figsize=(9, 5))

# Our connectome
our_degrees = [d for _, d in G.degree()]
ax.hist(our_degrees, bins=range(0, max(our_degrees)+2), alpha=0.6,
        label='Our synthetic connectome', color='steelblue', density=True)

# C. elegans approximation (log-normal fitted to known distribution)
# Mean degree ≈ 42; fit log-normal with mean=42
cel_mean   = celegans_stats['avg_total_degree']
cel_sigma  = 1.0  # approximate from literature
cel_mu     = np.log(cel_mean) - 0.5 * cel_sigma**2
cel_sample = np.random.lognormal(cel_mu, cel_sigma, 302)
ax.hist(cel_sample, bins=50, alpha=0.5, label='C. elegans (approx.)', color='tomato', density=True)

ax.set_xlabel('Total degree (in + out)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Degree Distribution Comparison', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('Key observations:')
print('  1. Scale: C. elegans has 302 neurons vs. our ~8. Properties only become')
print('     meaningful at scale (e.g., power-law distributions need many nodes).')
print('  2. Density: C. elegans has ~7% density — our sparse connectome is similar.')
print('  3. Clustering: C. elegans clustering (0.028) is higher than a random graph')
print('     of same density (~0.007), confirming small-world structure.')
print('  4. Path length: 2.65 hops in C. elegans is remarkably short for 302 nodes,')
print('     demonstrating efficient global communication via hub neurons.')
print('  5. Degree distribution: C. elegans follows a heavy-tailed distribution;')
print('     a few hub interneurons (e.g., AVA, AVB) have very high connectivity.')